# מפגש 18 — מחברת הדגמה Audited
סיפור: מיון מוצרים תקינים ופגומים; אזעקת שווא לעומת החמצת פגם. חיובי הוא Defect. שורות אמת, עמודות חיזוי. בשיעור זה עוסקים בספירות ו־Accuracy, לא ב־Precision/Recall/F1. 90 דקות הוראה ו־45 דקות פרויקט.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
actual=["OK"]*4+["Defect"]*4
predicted=["OK","OK","Defect","OK","OK","OK","Defect","Defect"]
labels=["OK","Defect"]

## א. ספירה ידנית
סמנו עבור שמונת הזוגות TN, FP, FN או TP וכתבו מראש מה יהיה התא הראשון בטבלה.

In [ ]:
cm=confusion_matrix(actual,predicted,labels=labels)
table=pd.DataFrame(cm,index=labels,columns=labels)
table.index.name="Actual"
table.columns.name="Predicted"
tn,fp,fn,tp=cm.ravel()
display(table)

In [ ]:
assert cm.tolist()==[[3,1],[2,2]]
assert cm.shape==(2,2)
assert len(actual)==len(predicted)==8
assert cm.sum()==8
assert (tn,fp,fn,tp)==(3,1,2,2)
assert tn+fp==4
assert fn+tp==4
assert tn+fn==predicted.count("OK")
assert fp+tp==predicted.count("Defect")
assert int(cm.trace())==sum(a==b for a,b in zip(actual,predicted))
assert accuracy_score(actual,predicted)==5/8
assert (tn+tp)/cm.sum()==accuracy_score(actual,predicted)
print("Binary checks pass")

## ב. מה אומר כל תא? חשבו מהו מספר החיזויים הנכונים ומהו סך כל המקרים.

In [ ]:
ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=labels).plot(values_format="d")
plt.show()

## ג. ניסוי קטן
שנו רק בעותק תחזית של מוצר תקין אחד מ־OK ל־Defect. חזו מה יקרה ל־FP ול־Accuracy.

In [ ]:
changed=predicted.copy()
changed[0]="Defect"
changed_cm=confusion_matrix(actual,changed,labels=labels)
assert changed_cm[0,1]==cm[0,1]+1
assert changed_cm[0,0]==cm[0,0]-1
assert changed_cm[1,0]==cm[1,0]
assert changed_cm[1,1]==cm[1,1]
assert changed_cm.sum()==cm.sum()
assert accuracy_score(actual,changed)<accuracy_score(actual,predicted)
print(changed_cm)

## ד. מעבר ל־Penguins
אותו קוד מקור של המודל המוכר: flipper_length_mm/body_mass_g, seed42, stratify, Test 25%, השלמה ותקנון רק על Train, K=3. התא הבא דורש CSV נגיש.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

## ה. מטריצה רב־מחלקתית
בחרו סדר Adelie, Chinstrap, Gentoo. פרשו תא Chinstrap→Adelie, סכום שורה, סכום עמודה ואלכסון.

In [ ]:
labels_species=["Adelie","Chinstrap","Gentoo"]
cm_species=confusion_matrix(y_test,predictions,labels=labels_species)
tab_species=pd.DataFrame(cm_species,index=labels_species,columns=labels_species)
display(tab_species)
assert cm_species.shape==(3,3)
assert cm_species.sum()==len(y_test)
assert (cm_species>=0).all()
assert cm_species.trace()==sum(y_test.to_numpy()==predictions)
assert abs(cm_species.trace()/cm_species.sum()-accuracy_score(y_test,predictions))<1e-12
assert cm_species[1,:].sum()==sum(y_test=="Chinstrap")
assert cm_species[:,1].sum()==sum(predictions=="Chinstrap")
ConfusionMatrixDisplay(confusion_matrix=cm_species,display_labels=labels_species).plot(values_format="d")
plt.show()

## כרטיס יציאה
איזו טעות חמורה יותר בבקרת איכות? הסבירו גם למה Accuracy לבדו לא קובע עלות. תכננו מטריצה בפרויקט ב־45 הדקות הייעודיות.